In [0]:
%sql
SELECT * FROM spb.silver_places LIMIT 100;

In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.spb.silver_places")

silver.select(
    F.countDistinct("basic_category").alias("n_basic")
).show()

silver.groupBy("basic_category").count().orderBy(F.desc("count")).show(30, truncate=False)

# places per cell: 10th, 50th and 90th percentile
silver.groupBy("h3_cell").count().select(
    F.expr("percentile_approx(count, array(0.1, 0.5, 0.9))").alias("places_per_cell")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F, Window

def cell_category_counts(df, cat_col="basic_category"):
    """One row per (cell, category) with its place count."""
    return (
        df.withColumn("category", F.coalesce(F.col(cat_col), F.lit("unknown")))
          .groupBy("h3_cell", "category")
          .agg(F.count("*").alias("n"))
    )

def places_per_cell(df):
        return counts.groupBy("h3_cell").agg(
            F.sum("n").alias("n_places"),
            F.count("*").alias("n_categories")
        )

def category_shares(counts):
        w = Window.partitionBy("h3_cell")
        return counts.withColumn("share", F.col("n") / F.sum("n").over(w))
    
def diversity(shares):
        "Shannon entropy in bits: -sum(p * log2 p). abs()"
        return shares.groupBy("h3_cell").agg(
            F.abs(-F.sum(F.col("share") * F.log2("share"))).alias("entropy_bits")
        )
    
def top_categories(counts, n=5):
        w = Window.partitionBy("h3_cell").orderBy(F.desc("n"), F.asc("category"))
        return counts.withColumn("category_rank", F.row_number().over(w)).where(F.col("category_rank") <= n)

In [0]:
silver = spark.table("workspace.spb.silver_places")

counts = cell_category_counts(silver)
shares = category_shares(counts)
stats = places_per_cell(counts).join(diversity(shares), "h3_cell")
top5 = top_categories(counts)

shares.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_cell_category")
stats.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_cell_stats")
top5.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_cell_top5")

# Check against silver
print("silver rows", silver.count())
print("sum of n in category tbl", spark.table("workspace.spb.gold_cell_category").agg(F.sum("n")).first()[0])
print("sum of n_places in stats", spark.table("workspace.spb.gold_cell_stats").agg(F.sum("n_places")).first()[0])
print("cells: silver vs stats", silver.select("h3_cell").distinct().count(), spark.table("workspace.spb.gold_cell_stats").count())

In [0]:
counts.display()

In [0]:
from pyspark.sql import functions as F, Window

H3_RES = 8

nodes_b = spark.table("workspace.spb.bronze_walk_nodes")
edges_b = spark.table("workspace.spb.bronze_walk_edges")

print(edges_b.join(nodes_b, edges_b.src == nodes_b.node_id, "left_anti").count())
print(edges_b.join(nodes_b, edges_b.dst == nodes_b.node_id, "left_anti").count())

print(nodes_b.count(), nodes_b.select("node_id").distinct().count())

silver_nodes = nodes_b.withColumn(
    "h3_cell", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES})")   # lon first
)
silver_nodes.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_walk_nodes")

In [0]:
K = 1
bars = spark.table("workspace.spb.silver_bars").select("id", "lon", "lat", "h3_cell")
nodes = (spark.table("workspace.spb.silver_walk_nodes")
         .select("node_id", F.col("lon").alias("n_lon"),
                 F.col("lat").alias("n_lat"), F.col("h3_cell").alias("n_cell")))

bars_cells = bars.withColumn("search_cell", F.explode(F.expr(f"h3_kring(h3_cell, {K})")))

In [0]:
cand = bars_cells.join(nodes, bars_cells.search_cell == nodes.n_cell)

In [0]:
R = 6371000.0   # earth radius in meters

def haversine_m(lat1, lon1, lat2, lon2):
    p1, p2 = F.radians(lat1), F.radians(lat2)
    a = (F.sin((p2 - p1) / 2) ** 2
         + F.cos(p1) * F.cos(p2) * F.sin(F.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R * F.asin(F.sqrt(a))

cand = cand.withColumn("snap_m", haversine_m(F.col("lat"), F.col("lon"),
                                             F.col("n_lat"), F.col("n_lon")))

In [0]:
w = Window.partitionBy("id").orderBy("snap_m")
snapped = (cand.withColumn("rn", F.row_number().over(w))
               .where("rn = 1")
               .select(F.col("id").alias("bar_id"), "node_id", "snap_m"))

In [0]:
# bars that found no node at all
print(bars.join(snapped, bars.id == snapped.bar_id, "left_anti").count())

# distribution of snap distances
snapped.select(F.expr("percentile_approx(snap_m, array(0.5, 0.9, 0.99, 1.0))")).show(truncate=False)

# the worst 20, to check on a map
snapped.orderBy(F.desc("snap_m")).show(20)

In [0]:
snapped.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.snapped_walk_nodes")

In [0]:
bn = spark.table("workspace.spb.snapped_walk_nodes")
print("bars:", bn.count(), "| distinct nodes:", bn.select("node_id").distinct().count())

# how many bars sit on each shared node
(bn.groupBy("node_id").count()
   .groupBy(F.col("count").alias("bars_on_node")).count()
   .orderBy("bars_on_node").show())

In [0]:
import numpy as np
from sklearn.neighbors import BallTree   # pip install scikit-learn if missing

nodes_pd = (spark.table("workspace.spb.silver_walk_nodes")
            .select("node_id", "lat", "lon").toPandas())

sample_pd = (spark.table("workspace.spb.snapped_walk_nodes")
             .join(spark.table("workspace.spb.silver_bars")
                        .select(F.col("id").alias("bar_id"), "lat", "lon"), "bar_id")
             .orderBy(F.rand(42)).limit(50).toPandas())

# BallTree's haversine metric wants (lat, lon) in RADIANS, in that order
tree = BallTree(np.radians(nodes_pd[["lat", "lon"]].to_numpy()), metric="haversine")
dist, ind = tree.query(np.radians(sample_pd[["lat", "lon"]].to_numpy()), k=1)

sample_pd["bt_node"] = nodes_pd["node_id"].to_numpy()[ind[:, 0]]
sample_pd["bt_m"] = dist[:, 0] * 6371000        # radians -> meters, same radius as before

print("same node:", (sample_pd.bt_node == sample_pd.node_id).mean())
print("max distance difference (m):", (sample_pd.bt_m - sample_pd.snap_m).abs().max())

In [0]:
big = bn.groupBy("node_id").count().where("count >= 5").select("node_id")
(bn.join(big, "node_id")
   .join(spark.table("workspace.spb.silver_bars")
              .select(F.col("id").alias("bar_id"), "name", "lat", "lon"), "bar_id")
   .orderBy("node_id").show(20, truncate=False))